Note: The script uses Berkeley Neural Parser to parse the generated instructions, and visualize the results using Plotly.

Please make sure to install benepar following their documentation [here](https://github.com/nikitakit/self-attentive-parser#installation).

In [1]:
import benepar, spacy
nlp = spacy.load('en_core_web_md')
doc = nlp("The time for action is now. It's never too late to do something.")

if spacy.__version__.startswith('2'):
    nlp.add_pipe(benepar.BeneparComponent("benepar_en3"))
else:
    nlp.add_pipe("benepar", config={"model": "benepar_en3"})

In [2]:
def find_root_verb_and_its_dobj(tree_root):
    # first check if the current node and its children satisfy the condition
    if tree_root.pos_ == "VERB":
        for child in tree_root.children:
            if child.dep_ == "dobj" and child.pos_ == "NOUN":
                return tree_root.lemma_, child.lemma_
        return tree_root.lemma_, None
    # if not, check its children
    for child in tree_root.children:
        return find_root_verb_and_its_dobj(child)
    # if no children satisfy the condition, return None
    return None, None

def find_root_verb_and_its_dobj_in_string(s):
    doc = nlp(s)
    first_sent = list(doc.sents)[0]
    return find_root_verb_and_its_dobj(first_sent.root)

find_root_verb_and_its_dobj_in_string("Write me a story about education.")

You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


c:\Users\1J1870897\Anaconda3\lib\site-packages\torch\distributions\distribution.py:45: UserWarning: <class 'torch_struct.distributions.TreeCRF'> does not define `arg_constraints`. Please set `arg_constraints = {}` or initialize the distribution with `validate_args=False` to turn off validation.
  warnings.warn(f'{self.__class__} does not define `arg_constraints`. ' +


('write', 'story')

In [3]:
import pandas as pd
import json
import tqdm

df = pd.read_csv('../test/genai_questions_1.csv')
instructions = df['questions'].to_list()

raw_phrases = []
for instruction in tqdm.tqdm(instructions):
    try:
        verb, noun = find_root_verb_and_its_dobj_in_string(instruction)
        raw_phrases.append({
            "verb": verb,
            "noun": noun,
            "instruction": instruction
        })
    except Exception as e:
        print(e)
        print(instruction)

  0%|          | 0/63 [00:00<?, ?it/s]

100%|██████████| 63/63 [00:20<00:00,  3.01it/s]


In [4]:
len(raw_phrases)

63

In [5]:
raw_phrases = pd.DataFrame(raw_phrases)
phrases = pd.DataFrame(raw_phrases).dropna()
phrases[["verb", "noun"]].groupby(["verb", "noun"]).size().sort_values(ascending=False)

verb        noun       
discuss     role           4
provide     detail         4
describe    process        3
explain     concept        2
perform     inspection     1
handle      collection     1
            failure        1
monitor     efficiency     1
            gearbox        1
perform     analysis       1
address     issue          1
discuss     use            1
provide     example        1
            information    1
            overview       1
use         analysis       1
validate    accuracy       1
prioritize  maintenance    1
discuss     system         1
affect      operation      1
discuss     source         1
            practice       1
            off            1
            mode           1
            importance     1
            benefit        1
determine   rate           1
describe    type           1
            technique      1
            system         1
            integration    1
            impact         1
            condition      1
validate    perform

In [6]:
top_verbs = phrases[["verb"]].groupby(["verb"]).size().nlargest(20).reset_index()

df = phrases[phrases["verb"].isin(top_verbs["verb"].tolist())]
# df = df[~df["noun"].isin(["I", "what"])]
# df = phrases
# df[~df["verb"].isin(top_verbs["verb"].tolist())]["verb"] = "other"
# df[~df["verb"].isin(top_verbs["verb"].tolist())]["noun"] = "other"
df = df.groupby(["verb", "noun"]).size().reset_index().rename(columns={0: "count"}).sort_values(by=["count"], ascending=False)
# df = df[df["count"] > 10]
df = df.groupby("verb").apply(lambda x: x.sort_values("count", ascending=False).head(4)).reset_index(drop=True)
df

,verb,noun,count
0,address,issue,1
1,affect,operation,1
2,describe,process,3
3,describe,type,1
4,describe,technique,1
5,describe,system,1
6,determine,rate,1
7,discuss,role,4
8,discuss,use,1
9,discuss,system,1


In [7]:

import plotly.graph_objects as go
import plotly.express as px

# df["blank"] = "ROOT"
# df = phrases.groupby(["verb", "noun"]).size().sort_values(ascending=False).head(5).reset_index().rename(columns={0: "count"})

df = df[df["count"] >= 1]
fig = px.sunburst(df, path=['verb', 'noun'], values='count')
# fig.update_layout(uniformtext=dict(minsize=10, mode='hide'))
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),
    font_family="Times New Roman",
)
fig.show()
fig.write_html("verb_noun.html")
#fig.savefig("output/verb_noun.pdf")

In [8]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

# Assuming df and phrases are defined elsewhere in your code
# df = df[df["count"] >= 1]

fig = px.sunburst(df, path=['verb', 'noun'], values='count')
fig.update_layout(
    margin=dict(l=0, r=0, t=0, b=0),
    font_family="Times New Roman",
)

# Save the plot as a .png file
pio.write_image(fig, 'verb_noun.png')

# Show the plot
fig.show()


In [14]:
!pip install -U kaleido

     -------------------------------------- 65.9/65.9 MB 743.1 kB/s eta 0:00:00


In [9]:
df['count'].sum()

35